# meeting_extracted_w1_t2_extract — W1 第三支：T2 L1 中立萃取（LLM）

- 用途：讀清洗表可萃取段（`inline_minutes` / `thread_reply`、`dup_of IS NULL`），每段呼叫一次 L1（規格 §5.1），程式驗證 / 正規化後寫 `g_meeting_items_{unit}`（規格 §4.2、§3.3；偏離處見 `../meeting_extracted_migration/STATUS.md` D22–D24）。
- 輸入：`s_meeting_record_clean_{unit}`、`s_meeting_participants_{unit}`（只取 token / side / company 給 prompt；姓名只在 driver 做 S3 掃描，不進 prompt、不印）、`s_semantic_entity_alias` / `s_semantic_entity`（customer 正規化）
- 輸出：`g_meeting_items_{unit}`（鍵 `item_id`；重萃不刪：舊條目 `superseded = true`、新條目 `supersedes_item_id`）；表由 002 建
- LLM：Azure Foundry Anthropic Messages 端點（同 `jobs/sales_report_mail`；secret `llm-api-keys/ms-foundry`）、`model` widget（預設 `claude-sonnet-5`）、temperature 0、只輸出 JSON、system prompt 開 prompt caching
- 參數（widgets）：`catalog`、`schema`、`run_mode`、`date_from` / `date_to`、`unit`、`prompt_version`、`model`、`max_segments`（費用閘門）、`concurrency`、`retry_failed`、`prompt_cache`、`dry_run`
- 排程：W1 每週一，接在 `meeting_extracted_w1_t1_clean` 之後（同一 Job 第 3 個 task）。首次 `run_mode = backfill`、`dry_run = true`、`max_segments = 5` 看樣本；再 `max_segments = 50` 看關卡；最後 `dry_run = false`、`max_segments = 0` 全量。
- Cell：[c01] params → [c02] 常數 → [c03] 共用純函式 → [c04] prompt → [c05] 後處理純函式 → [c06] LLM 接線 → [c07] DataFrame 函式 → [c10] 取本批 → [c11] 呼叫 → [c12] 關卡 → [c20] 寫表 → [c30] 驗證

Cell 標籤規則見 `docs/conventions.md`。


In [ ]:
# [c01] params
# W1 第三支：T2 L1 中立萃取。讀清洗表可萃取段 → 每段呼叫一次 LLM → 程式驗證 / 正規化 → 寫 g_meeting_items_{unit}。
# 由 job parameters 覆蓋；widget 建立過後改預設值不會生效，要重設先 dbutils.widgets.removeAll()。
from datetime import date, datetime

dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.dropdown("run_mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("date_from", "2026-06-01")   # backfill：對清洗表 mail_date，含
dbutils.widgets.text("date_to", "2026-12-31")
dbutils.widgets.text("unit", "bu")                 # 決定讀寫哪一組表（s_meeting_record_clean_{unit} …）
dbutils.widgets.text("prompt_version", "0.2")      # 要等於 [c04] PROMPT_VERSION_IN_CELL；改 prompt 就升，舊版本萃的段在 incremental 會重萃，舊條目標 superseded
dbutils.widgets.text("model", "claude-sonnet-5")   # Azure Foundry 上的模型名（同 jobs/sales_report_mail）
dbutils.widgets.text("max_segments", "20")         # 費用閘門：本次最多送幾段給 LLM；0 = 不限（backfill 全量時用）
dbutils.widgets.text("concurrency", "4")           # 同時打幾個 LLM 請求；看端點 rate limit 調
dbutils.widgets.dropdown("retry_failed", "false", ["true", "false"])  # true：上次 failed_* 的段也重萃
dbutils.widgets.dropdown("prompt_cache", "true", ["true", "false"])   # system prompt 開 prompt caching；端點拒絕就改 false
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])  # true：照樣呼叫 LLM（會花錢，受 max_segments 限制）、印關卡與樣本，不寫表


def _date(s: str) -> date:
    return datetime.strptime(s.strip(), "%Y-%m-%d").date()


cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "run_mode": dbutils.widgets.get("run_mode").strip(),
    "date_from": _date(dbutils.widgets.get("date_from")),
    "date_to": _date(dbutils.widgets.get("date_to")),
    "unit": dbutils.widgets.get("unit").strip(),
    "prompt_version": dbutils.widgets.get("prompt_version").strip(),
    "model": dbutils.widgets.get("model").strip(),
    "max_segments": int(dbutils.widgets.get("max_segments")),
    "concurrency": max(1, int(dbutils.widgets.get("concurrency"))),
    "retry_failed": dbutils.widgets.get("retry_failed") == "true",
    "prompt_cache": dbutils.widgets.get("prompt_cache") == "true",
    "dry_run": dbutils.widgets.get("dry_run") == "true",
}
assert cfg["catalog"] and cfg["schema"], "catalog / schema 不可為空"
assert cfg["date_from"] <= cfg["date_to"], "date_from 不可晚於 date_to"
assert cfg["unit"] and cfg["prompt_version"] and cfg["model"], "unit / prompt_version / model 不可為空"
assert cfg["max_segments"] >= 0 and cfg["concurrency"] <= 16, "max_segments ≥ 0；concurrency ≤ 16"
print(cfg)


In [ ]:
# [c02] imports
# [c02]～[c05]、[c07] 是純 Python / 純 DataFrame 函式（不碰 spark session / dbutils / 網路），tests/ 直接載入；
# [c06] 是 LLM 接線（secret、HTTP）。常數集中在這裡；受控詞表照規格 §6.1 / §6.2。
import hashlib
import json
import re
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor
from datetime import date, datetime

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

STATEMENT_TYPES = ("fact", "plan", "request", "decision", "issue", "opinion", "definition")
TOPIC_TAGS = ("demand", "allocation", "price", "cost", "product_plan", "spec", "schedule", "supply_chain",
              "capacity", "quality", "engineering", "commercial_relation", "terminology", "admin")
SIDES = ("auo", "customer", "third_party", "unknown")
APPS = ("TV", "NB", "MNT", "ALL")
APP_BASIS = ("model_prefix", "spec_context", "customer_default", "subject", "unknown")
DATE_BASIS = ("segment_header", "inline_date", "mail_date_fallback")
EXTRACT_STATUS = ("ok", "failed_schema", "failed_quote", "empty")  # empty：D23，段落沒有可收條目的標記列
EXTRACTABLE_TYPES = ("inline_minutes", "thread_reply")
MAX_USER_CHARS = 12_000        # §5.1 user ≤ 12k 字元；超過仍整段送，只記 log（§11.1 sub_no 延後）
MAX_ITEMS_PER_SEGMENT = 40     # 防 LLM 失控：超過視為整段 failed_schema
MAX_TAGS = 3                   # §6.2 topic_tags 1–3 個
MAX_FREE_TAGS = 10
SPEAKERLESS_CONF_CAP = 0.7     # §6.7 紀要格式（speaker_token NULL）confidence 上限
TOKEN_RE = re.compile(r"^[PG]\d{2,}$")
EMAIL_ANY_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+(?:\.[A-Za-z0-9\-]+)+")
MODEL_RE = re.compile(r"\b([BMT]\d{3}[A-Z]{3}\d{2}\.[A-Z0-9]+)")
PREFIX_APP = {"B": "NB", "M": "MNT", "T": "TV"}
# §6.6 敏感度關鍵字（程式判）：S2 命中優先於 S1；都沒有 → S0。「K」只在數字後面才算（10K、50 K）。
S2_RE = re.compile(
    r"allocation|限簽|\bd?ppm\b|US\$|\$\s*\d|報價|rebate|incentive|\bdeal\b|結價|\bMDF\b|\d\s*K(?:pc)?\b", re.I
)
S1_RE = re.compile(r"\bBP\b|\bFCST\b|\bLTF\b|一定盈|策略|方向|\bTAM\b", re.I)
# LLM 費用估算用（§10.1：Sonnet 5 標價，美元 / 百萬 token；cache 寫 1.25×、讀 0.1×）。只印 log，不入表。
PRICE_PER_MTOK = {"input": 2.0, "output": 10.0, "cache_write": 2.5, "cache_read": 0.2}

# 從清洗表讀的欄（送 driver 逐段呼叫 LLM）；參與者表每列取的欄（只到 driver 做 token 清單與 S3 掃描，不印）
SEG_COLS = ["pk_seq", "segment_no", "sub_no", "unit", "create_date", "mail_date", "subject_clean", "segment_date",
            "date_inferred", "content_type", "parties", "customer_hint", "app_hint", "clean_text", "clean_len",
            "cleaner_version"]
PART_COLS = ["pk_seq", "token", "role", "kind", "side", "company", "display_name_raw", "name_zh", "name_en"]


In [ ]:
# [c03] pure_shared
# 與 w1_t1_mail [c03] / w1_t1_clean [c03] 相同的純函式（norm_space、normalize_text、sha1_hex）。
# 複製而不是 import：Databricks 只能貼 cell。改一邊要同步其他兩支；tests 有一條比對三支結果一致。
def norm_space(s: str | None) -> str:
    """所有空白（含 U+00A0、換行）壓成一格並去頭尾。"""
    return re.sub(r"\s+", " ", (s or "").replace("\xa0", " ")).strip()


def normalize_text(s: str | None) -> str:
    """去重 / 穩定鍵共用的正規化：NFKC（全半形統一）、去標點、去空白、去控制字元、小寫。"""
    t = unicodedata.normalize("NFKC", s or "")
    return "".join(ch for ch in t if unicodedata.category(ch)[0] not in "PZC").lower()


def sha1_hex(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()


In [ ]:
# [c04] prompt_l1
# L1 中立萃取 prompt（規格 §5.1 十個構件；§6.1 / §6.2 / §6.5 / §6.7 規則；附錄 A 當 few-shot）。prompt 內嵌成常數（D2）。
# 任何 prompt 都不帶姓名 / email：人只以 token ＋ side ＋ 公司出現（§5.0）。改這個 cell 就升 [c01] prompt_version。
# 詞彙表是 PM_Head business_semantics/會議記錄萃取素材.md 的壓縮版（只留術語，不帶市場數字）。
PROMPT_VERSION_IN_CELL = "0.2"  # [c11] 會比對 cfg["prompt_version"]，不一致就停（§2.4 第 4 點）
# 0.1 → 0.2（2026-10-16）：summary 不得帶未 token 化的人名（改寫成角色）、quote 挑不含人名的片段；50 段 dry_run 看到英文名被抄進 summary。

GLOSSARY_COMPACT = """【詞彙表】只用來辨識 terms_mentioned 與正確理解縮寫；summary 不必展開縮寫。
銷售需求：BP 年度/季度出貨目標；FCST 滾動預測（常每週）；LTF 長期預測；一定盈 內部最低信心預測；Billing / Billing QTY 實際開票出貨量；Pmix 產品組合占比；TAM / TAM share AUO 在客戶面板採購的份額；Deal 特殊數量折扣（季談，LTA 之外）；LTA 年度合約含量保證與 rebate；Pull-in 需求前移；Roll over / SB 訂單延到下期；DSI 客戶側庫存天數；Risk buy / 策備 無正式訂單先備料；限簽 訂單上限；Allocation / 排程 月度生產分配。
價格成本：ASP 平均售價；Cost adder 轉產地附加費；Price parity 與對手價格對齊；結價 月度議價確認；MDF 市場開發基金；rebate / incentive 回饋。
開發認證：RFQ 正式詢價；RFI 前期規格探詢；BA 客戶核准量產投片；EC 工程變更；EOL 停產；NPI 新品導入；MRL 製造成熟度；Mask priority 新光罩開案優先序；Tape out 光罩送製；MP 量產；OC（Open Cell）裸面板；LCM 含背光模組；PCBA 電路板組件；Set / 整機 終端成品；ODM 代工（Foxconn / Quanta / Qisda / Compal）。
客戶角色：GCM 品類採購主管（Dell / Lenovo）；GSM 供應管理；DGP Dell 面板採購；SDE / PJM Dell 技術 / 專案；SCM / PROC Samsung VD 供應鏈 / 採購；SQE 供應商品質工程。
規格：尺寸 11.6–17.3" 多為 NB、23.8–34" 多為 MNT、32" 以上多為 TV；a-Si / LTPS / Oxide（IGZO）/ OLED 技術平台；Resolution HD / FHD / FHD+ / QHD / UHD / 2.8K / 3K2K / 5K；Refresh 60–360Hz，HSR 高速倍頻；Touch OTP / iTP / add-on；Color gamut NTSC / sRGB / DCI-P3；nit 亮度；POL 偏光片（BMC / SDI / AG2 / AG25 等）；DFR 動態刷新；LVDS / eDP 介面；BOM 物料表。
型號：B140UAN08.5（B = NB / MNT 面板，140 = 14.0"）、M270QAN06.A（M = MNT，270 = 27"）、T430QVN04.5（T = TV，430 = 43"）。
廠區排程：L6K 昆山 G6（LTPS）；L6B 后里 G6；7A / 7B / 8A / 8B G7.5 / G8.5 廠；S02 / S11 後段模組廠；M11 模組線；JI-in / JI-out 模組投入 / 產出；Cell-in 前段投入；Array TFT 製程；SKDL 排程；Demura Mura 補償製程。
物流：ETD / ETA 離港 / 到港；Charter 包機包船；Hub 客戶側倉；SAMEX / SEHC Samsung 墨西哥 / 韓國廠；VSL 越南海運。
品質：RA 可靠性驗證（T100…T2000 累計時數）；ND Mura 可見度閾值；JND 人眼可感知差異；Flicker 閃屏 dB；VLRR 可變低刷新率不良；Mura 不均；IS 殘影；Cold start NG 低溫啟動失敗；ppm / dppm 不良率。
對手：BOE 京東方；CSOT 華星（TCL 集團）；HKC 惠科；LGD LG Display；INNX 群創；SDI Samsung Display / SDI；Tianma 天馬；Sharp 夏普。
時間：CY 曆年；FY 財年（Dell 自 2 月起算，FY27 Q1 = 2026/2–4）；FY25/26 Lenovo 跨年財年；Y26 / Y27 產品代次；H1 / H2 上下半年；B / M / E 月初 / 中 / 底；wk1–wk5 月內週；CS26 / CS27 客戶 Spec 年度。
地緣：Vietnam PCBA 非中國產電路板；DAO Dell 美洲營運；DPN Dell 料號；Non-China 非中國供應定位；USMCA 美墨加協定。
單位：Kpc 千片；K 千（台 / 片）；M 百萬；USD / 片；DSI 天；ppm / dppm。"""

FEW_SHOT = """【輸出示例】（取自其他段落，只示範拆法與欄位用法；示例中的日期與數字與本段無關）
紀要段「T850QVN08.LT0：LT0 BOM 已升生效 (SMK POL AG25料號: 57.85T25.091)…POL SMK AG25 請採購先備10K原材…」→ 拆成兩條：
{"statement_type": "decision", "speaker_token": null, "speaker_side": "unknown", "customer_raw": "TCL", "app": "TV", "app_basis": "model_prefix", "model_nos": ["T850QVN08.LT0"], "topic_tags": ["product_plan", "spec"], "free_tags": ["AG25", "SMK", "POL"], "summary": "T850QVN08.LT0 的 BOM 已升版生效，改用 SMK AG25 偏光板", "quote": "LT0 BOM 已升生效 (SMK POL AG25料號: 57.85T25.091)", "event_date": "2026-09-03", "date_basis": "segment_header", "time_refs": [{"when": "2026-10", "what": "目標放量"}], "numbers": [], "terms_mentioned": ["BOM", "POL"], "confidence": 0.9}
{"statement_type": "request", "speaker_token": null, "speaker_side": "auo", "customer_raw": "TCL", "app": "TV", "app_basis": "model_prefix", "model_nos": ["T850QVN08.LT0"], "topic_tags": ["supply_chain"], "free_tags": ["AG25", "原材"], "summary": "請採購先備 SMK AG25 偏光板原材 10K，入料 10/14～10/26", "quote": "POL SMK AG25 請採購先備10K原材", "event_date": "2026-09-03", "date_basis": "segment_header", "time_refs": [{"when": "2026-10-14~2026-10-26", "what": "入料"}], "numbers": [{"value": 10, "unit": "K", "what": "POL 原材備料"}], "terms_mentioned": ["POL"], "confidence": 0.85}
紀要段「T430QVN04.5：量產出貨VLRR不良目前603 ppm…TCL要求400 dppm以下」→ 事實與要求拆開：
{"statement_type": "fact", "speaker_token": null, "speaker_side": "unknown", "customer_raw": "TCL", "app": "TV", "app_basis": "model_prefix", "model_nos": ["T430QVN04.5"], "topic_tags": ["quality"], "free_tags": ["VLRR"], "summary": "T430QVN04.5 量產出貨 VLRR 不良目前 603 ppm，Fab 持續改善 particle 與 leakage 攔檢", "quote": "量產出貨VLRR不良目前603 ppm", "event_date": "2026-09-03", "date_basis": "segment_header", "time_refs": [], "numbers": [{"value": 603, "unit": "ppm", "what": "VLRR 不良"}], "terms_mentioned": ["VLRR", "ppm"], "confidence": 0.9}
{"statement_type": "request", "speaker_token": null, "speaker_side": "customer", "customer_raw": "TCL", "app": "TV", "app_basis": "model_prefix", "model_nos": ["T430QVN04.5"], "topic_tags": ["quality"], "free_tags": ["VLRR"], "summary": "TCL 要求 VLRR 不良降到 400 dppm 以下", "quote": "TCL要求400 dppm以下", "event_date": "2026-09-03", "date_basis": "segment_header", "time_refs": [], "numbers": [{"value": 400, "unit": "dppm", "what": "TCL 要求上限"}], "terms_mentioned": ["dppm"], "confidence": 0.9}
工程細節照樣收（中立層不篩）：
{"statement_type": "fact", "speaker_token": null, "speaker_side": "unknown", "customer_raw": "TCL", "app": "TV", "app_basis": "model_prefix", "model_nos": ["T550QVN10.F"], "topic_tags": ["engineering", "capacity"], "free_tags": ["S11", "UV膠"], "summary": "S11 於 8/29 做藍色 UV 膠小批量驗證，20 片中 3 片 NG，待確認是否可滿足", "quote": "S11 8/29 小批量驗證結果: 20片3片NG", "event_date": "2026-08-29", "date_basis": "inline_date", "time_refs": [], "numbers": [{"value": 3, "unit": "片", "what": "NG 數（20 片中）"}], "terms_mentioned": [], "confidence": 0.8}
thread 段（<P03>: 開頭、Sent: 2026-08-13）→ 發言人以 token 指涉：
{"statement_type": "plan", "speaker_token": "P03", "speaker_side": "auo", "customer_raw": "acer", "app": "MNT", "app_basis": "model_prefix", "model_nos": ["M270QAN06.A"], "topic_tags": ["product_plan", "spec"], "free_tags": ["LVDS", "UHD", "100Hz"], "summary": "<P03> 說明 M270QAN06.A 為 27 吋 UHD 100Hz、4ch LVDS 開放式機種，outline 同 M270QAN06.0，MP 時程 Jan'27", "quote": "AUO Model Name: M270QAN06.A | Interface: 4ch LVDS | MP: Jan'27", "event_date": "2026-08-13", "date_basis": "segment_header", "time_refs": [{"when": "2027-01", "what": "MP"}], "numbers": [], "terms_mentioned": ["MP", "OC"], "confidence": 0.8}"""

SYSTEM_PROMPT_L1 = f"""你是會議紀錄的中立萃取器。輸入是一段已去識別的會議紀錄（thread 回覆或紀要的一個區塊），人名都已換成 <P01>、<G01> 這類 token。你的工作是把這段裡「說了什麼」拆成原子條目，輸出 JSON。

【中立紀律】
你只記錄會議裡說了什麼，不判斷它對誰有用、重不重要。不要因為內容是工程細節就略過，也不要因為內容像市場訊息就放大。不做任何「這對 X 有沒有用」的判斷、不評重要性、不分類成市場情報 / 事件，也不要嘗試還原任何人的姓名。

【收錄規則】
凡有實質資訊的陳述都收：數字、時程、規格、要求、決定、問題、看法、口徑說明。只排除：寒暄、會議行政（下次開會時間、已更新紀要、請查收）、與本段其他條目完全重複的內容。整段沒有可收的內容時 items 給空陣列，原因寫在 segment_note。

【原子化】
一條 = 一個對象（客戶、型號或議題）＋ 一個主張。同一句含兩個主張（例如「現況 X，客戶要求 Y」）就拆兩條。同一主張涵蓋多個型號可以放在一條的 model_nos。

【statement_type，一條只能一個】
fact 已發生或現況｜plan 一方的規劃或時程｜request 一方對另一方的要求｜decision 已拍板的選擇｜issue 待解問題或風險｜opinion 判斷或展望｜definition 對術語、口徑、規則的說明。

【topic_tags，只能從下表選 1–3 個，第一個放最能代表主張的；其他關鍵字放 free_tags】
demand（BP、FCST、需求上下修、拉貨、市佔）｜allocation（Allocation、限簽、出貨餘量、備貨量）｜price（報價、結價、Deal、rebate、MDF、incentive）｜cost（材料或製造成本增減、cost adder）｜product_plan（新機種、開案、停案、EOL、line-up）｜spec（解析度、刷新率、介面、POL、IC 選用）｜schedule（樣品、MP、驗證時程）｜supply_chain（缺料、交期、產地、供應商限制）｜capacity（廠別、轉廠、線體、產能）｜quality（不良率、ppm、RA、品質事件與改善）｜engineering（實驗條件、製程參數、改機）｜commercial_relation（合約、合資、份額、競爭對手動態、組織變動）｜terminology（術語、口徑、換算規則）｜admin（會議行政，原則上不收，收了就標這個）。

【發言人】
段落以「<Pnn>:」開頭者，該段的發言人就是 Pnn；句中「<P03> 表示…」則那句的發言人是 P03。speaker_token 只填 token 本身（例 "P03"，不含角括號）。紀要格式（沒有發言人標頭）判不出 → speaker_token 給 null，speaker_side 依語句主詞猜一個（「TCL 要求…」→ customer；「請採購…」→ auo；判不出 → unknown）並把 confidence 壓低。轉寄人與群組（Gnn）不能當發言人；使用者會告訴你哪些 token 是轉寄人 / 群組。speaker_side 只能是 auo / customer / third_party / unknown。

【客戶與應用】
customer_raw 填素材原名（VD、海信、acer），不要正規化。app 只能是 TV / NB / MNT / ALL / null；app_basis 依序：型號前綴（B → NB、M → MNT、T → TV）= model_prefix；規格語境（27"/31.5" + Hz → MNT；43"～98" → TV；14"～16" → NB）= spec_context；主旨關鍵字 = subject；客戶預設 = customer_default；都不成立 → ALL（客戶已知但 app 不明）或 null，app_basis = unknown。使用者給的「程式推測」只是參考，以段落內容為準。

【日期】
event_date 是陳述描述的事發生或被說的日期（yyyy-MM-dd）：thread 回覆取該段 Sent: 日期（date_basis = segment_header）；紀要取使用者給的段落日期（segment_header）；段內另有已發生事件的明確日期（「8/27 TCL 提供終簽」）→ 用該日期（inline_date）。未來時程（「預計 10 月 Move in」）不當 event_date，寫進 time_refs。判不出就填段落日期。

【quote】
逐字取自正文、取能支撐主張的最短片段、不跨段、不改字、不補字；可含 <Pnn> token；正文裡的「 | 」是表格攤平符號，要保留。quote 會用程式比對正文，對不上整條作廢。

【summary】
一句乾淨版，繁體中文，型號照原文，數字單位照原文。人只能以 <Pnn> 指涉，不得出現姓名或 email。

【人名（嚴格）】
正文裡若還留有沒換成 token 的人名（英文名如 Luke、中文姓名、「姓, 名」），summary 一律改寫成角色或省略（例：「分享給 MNT 成員（Luke, Juno）」→「分享給 MNT 成員」）；quote 要挑不含人名的片段，挑不到就把該條的 quote 縮到人名之前或之後的最短片段。公司名、部門代碼、產品與型號不是人名，照寫。

{GLOSSARY_COMPACT}

【輸出】只輸出一個 JSON 物件，不要 markdown 圍欄、不要說明文字：
{{"segment_note": "（整段備註，可空字串）",
  "items": [{{"statement_type": "", "speaker_token": null, "speaker_side": "", "customer_raw": null, "app": null, "app_basis": "",
             "model_nos": [], "topic_tags": [], "free_tags": [], "summary": "", "quote": "",
             "event_date": "yyyy-MM-dd", "date_basis": "", "time_refs": [{{"when": "", "what": ""}}],
             "numbers": [{{"value": 0, "unit": "", "what": ""}}], "terms_mentioned": [], "confidence": 0.0}}]}}

{FEW_SHOT}"""


def participants_block(pk_tokens: dict) -> str:
    """{token: {side, company, kind, is_forwarder}} → 給 LLM 的參與者清單；只有 token / 陣營 / 公司，沒有姓名。"""
    if not pk_tokens:
        return "（本信沒有解析到參與者）"
    lines = []
    for tok in sorted(pk_tokens):
        p = pk_tokens[tok]
        company = p.get("company") or "?"
        if p.get("kind") == "group":
            lines.append(f"- {tok}: 群組，{p.get('side') or 'unknown'} / {company}（不是發言人）")
        elif p.get("is_forwarder"):
            lines.append(f"- {tok}: {p.get('side') or 'unknown'} / {company}（轉寄人，不是發言人）")
        else:
            lines.append(f"- {tok}: {p.get('side') or 'unknown'} / {company}")
    return "\n".join(lines)


def build_user_prompt(seg: dict, pk_tokens: dict) -> str:
    """一段 → user prompt。seg 是清洗表一列（dict）；pk_tokens 見 participants_block。不帶姓名、email。"""
    basis = "以信件日期代替（推測）" if seg.get("date_inferred") else (
        "thread 標頭 Sent:" if seg.get("content_type") == "thread_reply" else "紀要 ITEM 日期")
    parties = ", ".join(seg.get("parties") or []) or "（無）"
    return (
        "## 段落資訊\n"
        f"- 主旨: {norm_space(seg.get('subject_clean')) or '（無）'}\n"
        f"- 段落日期（segment_date）: {seg.get('segment_date')}（{basis}）\n"
        f"- 段落類型: {seg.get('content_type')}\n"
        f"- 程式推測客戶（僅參考）: {seg.get('customer_hint') or '無'}；程式推測應用（僅參考）: {seg.get('app_hint') or '無'}\n"
        f"- 出現的組織網域: {parties}\n"
        "## 參與者（token / 陣營 / 公司）\n"
        f"{participants_block(pk_tokens)}\n"
        "## 正文（clean_text；quote 必須逐字取自此處）\n"
        "<<<\n"
        f"{seg.get('clean_text') or ''}\n"
        ">>>"
    )


COMPACT_HINT = (
    "\n\n【輸出長度限制】上一輪輸出太長被截斷。請精簡：quote 取能支撐主張的最短片段（通常 ≤ 60 字）；summary 一句；"
    "同一型號的同一主張合併成一條；time_refs / numbers 只列段內明確寫出的；free_tags 最多 5 個。"
)


def retry_user_prompt(error: str) -> str:
    """JSON / schema 失敗後的第二輪 user 訊息（接在上一輪 assistant 輸出之後）。"""
    return f"上一輪輸出無法使用：{error}。請重新只輸出一個合法的 JSON 物件（同樣的 schema），不要任何說明文字或 markdown 圍欄。"


In [ ]:
# [c05] pure_postprocess
# LLM 回覆 → 條目列（規格 §4.2 後處理 1–9）：JSON 解析、schema 驗證、quote 子字串驗證、item_key、受控詞表過濾、
# 發言人以參與者表為準、customer 經別名表正規化、敏感度關鍵字、日期補值、S3 掃描。
# 失敗不丟：failed_schema / failed_quote 照樣寫列（View 只看 ok）；S3 命中的列清空 summary / quote 再寫。
def parse_llm_json(text: str | None) -> tuple[dict | None, str | None]:
    """回覆文字 → (dict, None) 或 (None, 錯誤訊息)。容忍 markdown 圍欄與前後說明文字（取第一個 { 到最後一個 }）。"""
    if not text or not text.strip():
        return None, "empty response"
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.I | re.M)
    start, end = t.find("{"), t.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None, "no JSON object found"
    try:
        obj = json.loads(t[start:end + 1])
    except json.JSONDecodeError as e:
        return None, f"invalid JSON: {e.msg} at pos {e.pos}"
    if not isinstance(obj, dict) or not isinstance(obj.get("items"), list):
        return None, "top-level must be an object with items: []"
    if len(obj["items"]) > MAX_ITEMS_PER_SEGMENT:
        return None, f"too many items ({len(obj['items'])} > {MAX_ITEMS_PER_SEGMENT})"
    return obj, None


def _ws_free(s: str | None) -> str:
    return re.sub(r"\s+", "", s or "")


QUOTE_SPLIT_RE = re.compile(r"\.{3,}|…+|⋯+")  # quote 用「…」/「...」串接同段內不連續片段（附錄 A 的寫法）
def _loose_norm(s: str | None) -> str:
    """第三層：NFKC、去標點與符號（** ~~ | + :）、去空白、小寫；字母與數字必須仍逐字相同。"""
    t = unicodedata.normalize("NFKC", s or "")
    return "".join(ch for ch in t if unicodedata.category(ch)[0] not in "PSZC").lower()


_QUOTE_NORMS = (
    _ws_free,                                                 # 1. 只去空白（§4.2 (2)）
    lambda s: unicodedata.normalize("NFKC", _ws_free(s)),     # 2. 全半形統一
    _loose_norm,                                              # 3. 去標點 / 符號 / md 殘渣 / 小寫
)


def quote_in_text(quote: str | None, clean_text: str | None) -> bool:
    """§4.2 (2) 子字串驗證，三層由嚴到寬：去空白 → NFKC → normalize_text（LLM 抄 quote 常自動清掉 ** 殘渣或改標點，
    字沒變就算過）。quote 可用「…」串接片段：每片依序都要在正文找到（正規化後 < 4 字的片段不計，但至少要有一片）。"""
    if not quote or not quote.strip():
        return False
    pieces = [p for p in QUOTE_SPLIT_RE.split(quote) if p.strip()]
    if not pieces:
        return False
    for norm in _QUOTE_NORMS:
        text, pos, checked, ok = norm(clean_text), 0, 0, True
        for piece in pieces:
            q = norm(piece)
            if not q or (len(q) < 4 and len(pieces) > 1):
                continue
            i = text.find(q, pos)
            if i == -1:
                ok = False
                break
            pos, checked = i + len(q), checked + 1
        if ok and checked:
            return True
    return False


def item_key_of(pk_seq: int, segment_no: int, quote: str | None) -> str:
    """穩定鍵 sha1(pk ‖ seg ‖ normalize(quote))：重萃後同一句話不變（§3.3）。"""
    return sha1_hex(f"{pk_seq}|{segment_no}|{normalize_text(quote)}")


def item_id_of(unit: str, pk_seq: int, segment_no: int, sub_no: int, nnn: int) -> str:
    return f"{unit.upper()}-MTG-{pk_seq}-{segment_no}-{sub_no}-{nnn:03d}"


def sensitivity_of(*texts: str | None) -> str:
    """§6.6 關鍵字規則：S2 > S1 > S0。S3 不在這裡判（S3 = 掃描失敗整條作廢）。"""
    blob = " ".join(t for t in texts if t)
    if S2_RE.search(blob):
        return "S2"
    if S1_RE.search(blob):
        return "S1"
    return "S0"


def _is_ascii(s: str) -> bool:
    return all(ord(c) < 128 for c in s)


def build_alias_index(aliases: list[tuple[str, str]]) -> tuple[dict, list[tuple[str, str, re.Pattern]]]:
    """[(alias, entity_id)] → (精確查表 {normalize(alias): entity_id}, 長到短的 (normalize(alias), entity_id, 整字 regex) 供包含比對)。"""
    exact: dict[str, str] = {}
    raw_of: dict[str, str] = {}
    for alias, eid in aliases:
        k = normalize_text(alias)
        if k and k not in exact:
            exact[k] = eid
            raw_of[k] = norm_space(alias)
    ordered = []
    for k, eid in sorted(exact.items(), key=lambda kv: (-len(kv[0]), kv[0])):
        rx = re.compile(rf"(?<![A-Za-z0-9]){re.escape(raw_of[k])}(?![A-Za-z0-9])", re.I)
        ordered.append((k, eid, rx))
    return exact, ordered


def normalize_customer(raw: str | None, exact: dict, ordered: list[tuple[str, str, re.Pattern]]) -> str | None:
    """customer_raw → entity_id：先精確（normalize 後相等），再「別名是原名的子字串」長的先吃（Samsung VD 先於 Samsung）。
    短 ASCII 別名（< 3 字，例 VD）只做整字比對（前後不接字母數字），避免吃掉 DVD。對不上 NULL（§4.2 (6)：LLM 給的正規化名一律以表為準）。"""
    k = normalize_text(raw)
    if not k:
        return None
    if k in exact:
        return exact[k]
    raw_s = norm_space(raw)
    for alias, eid, rx in ordered:
        if (len(alias) >= 3 or not _is_ascii(alias)) and alias in k:
            return eid
        if len(alias) < 3 and _is_ascii(alias) and rx.search(raw_s):
            return eid
    return None


def _clean_token(tok) -> str | None:
    if not isinstance(tok, str):
        return None
    t = tok.strip().strip("<>").upper()
    return t if TOKEN_RE.match(t) else None


def resolve_speaker(token, pk_tokens: dict) -> tuple[str | None, str | None, str | None]:
    """§6.7：回傳 (speaker_token, side_from_table, drop_reason)。token 不在本信清單 / 轉寄人 / 群組 → (None, None, 原因)。"""
    t = _clean_token(token)
    if token in (None, "") or (isinstance(token, str) and not token.strip()):
        return None, None, None
    if t is None or t not in pk_tokens:
        return None, None, "unknown_token"
    p = pk_tokens[t]
    if p.get("kind") == "group":
        return None, None, "group"
    if p.get("is_forwarder"):
        return None, None, "forwarder"
    return t, (p.get("side") if p.get("side") in SIDES else "unknown"), None


def name_variants_of(parts: list[dict]) -> set[str]:
    """S3 掃描用：本信所有參與者的姓名寫法（原字串、中文名、英文名、「姓, 名」）。ASCII ≥ 3 字、中文 ≥ 2 字。只在 driver 記憶體，不印。"""
    out: set[str] = set()
    for p in parts:
        if p.get("kind") == "group":
            continue
        cand = {p.get("display_name_raw"), p.get("name_zh"), p.get("name_en")}
        en = norm_space(p.get("name_en"))
        words = en.split(" ")
        if len(words) >= 2:
            cand.add(f"{words[-1]}, {' '.join(words[:-1])}")
        for s in cand:
            s = norm_space(s)
            if not s:
                continue
            ascii_ = all(ord(c) < 128 for c in s)
            if (ascii_ and len(s) >= 3) or (not ascii_ and len(s) >= 2):
                out.add(s)
    return out


def first_name_res(parts: list[dict]) -> list[re.Pattern]:
    """S3 掃描補強：參與者英文名的第一個字（≥ 3 字母）做整字、區分大小寫比對（Luke 抓得到，willing 不會被 Will 誤殺）。
    D17 讓 clean_text 留著這種稱呼，LLM 可能抄進 summary。"""
    firsts: set[str] = set()
    for p in parts:
        if p.get("kind") == "group" or not p.get("name_en"):
            continue
        first = norm_space(p["name_en"]).split(" ")[0].strip(",")
        if len(first) >= 3 and all(ord(c) < 128 for c in first) and first[0].isupper():
            firsts.add(first)
    return [re.compile(rf"(?<![A-Za-z]){re.escape(f)}(?![A-Za-z])") for f in sorted(firsts)]


def s3_hits(texts: list[str | None], names: set[str], first_res: list[re.Pattern] = ()) -> int:
    """summary / quote 含 email 樣式、參與者姓名（不分大小寫子字串）或參與者英文名首字（整字）的次數（§4.2 (9)）。"""
    blob = "\n".join(t for t in texts if t)
    n = len(EMAIL_ANY_RE.findall(blob))
    low = blob.lower()
    for name in names:
        if name.lower() in low:
            n += 1
    for rx in first_res:
        if rx.search(blob):
            n += 1
    return n


def _str_or_none(v) -> str | None:
    if isinstance(v, str):
        v = v.strip()
        return v or None
    return None


def _str_list(v, cap: int) -> list[str]:
    if not isinstance(v, list):
        return []
    out, seen = [], set()
    for x in v:
        if isinstance(x, (str, int, float)):
            s = str(x).strip()
            if s and s not in seen:
                seen.add(s)
                out.append(s)
    return out[:cap]


def _json_list(v) -> str:
    return json.dumps(v if isinstance(v, list) else [], ensure_ascii=False)


def _parse_date(v) -> date | None:
    if isinstance(v, str) and re.match(r"^\d{4}-\d{2}-\d{2}$", v.strip()):
        try:
            return datetime.strptime(v.strip(), "%Y-%m-%d").date()
        except ValueError:
            return None
    return None


def _app_from_models(models: list[str]) -> str | None:
    apps = {PREFIX_APP.get(m[0].upper()) for m in models if m and MODEL_RE.match(m)}
    apps.discard(None)
    return apps.pop() if len(apps) == 1 else None


def build_item_rows(resp: dict | None, error: str | None, seg: dict, ctx: dict) -> tuple[list[dict], dict]:
    """一段的 LLM 結果 → 條目 dict 列（欄位 = g_meeting_items 表，supersedes_item_id / extracted_at 由 Spark 補）＋ 統計。
    ctx：pk_tokens、names / first_names（S3 掃描）、alias_exact / alias_ordered、prompt_version、model、nnn_offset（既有條目的最大流水號）。
    resp None（兩輪都解析失敗）→ 一列 failed_schema（summary 放錯誤訊息）；items 空 → 一列 empty（summary 放 segment_note，D23）。"""
    st = {"items": 0, "ok": 0, "failed_schema": 0, "failed_quote": 0, "empty": 0, "s3": 0, "speaker_dropped": {},
          "tags_moved": 0, "no_topic": 0, "customer_null": 0, "date_fallback": 0}
    pk, seg_no, sub_no = seg["pk_seq"], seg["segment_no"], seg["sub_no"]
    base = {
        "pk_seq": pk, "segment_no": seg_no, "sub_no": sub_no, "unit": seg["unit"],
        "subject_clean": seg.get("subject_clean"), "superseded": False, "supersedes_item_id": None,
        "prompt_version": ctx["prompt_version"], "model": ctx["model"], "cleaner_version": seg.get("cleaner_version"),
        "extracted_at": None,
    }
    nnn = int(ctx.get("nnn_offset") or 0)

    def _row(n: int, **kw) -> dict:
        r = dict(base, item_id=item_id_of(seg["unit"], pk, seg_no, sub_no, n), item_key=None, statement_type=None,
                 speaker_token=None, speaker_side=None, customer_raw=None, customer=None, app=None, app_basis=None,
                 model_nos=[], topic_tags=[], topic_tag_primary=None, free_tags=[], summary=None, quote=None,
                 event_date=None, event_month=None, date_basis=None, time_refs="[]", numbers="[]",
                 terms_mentioned=[], sensitivity="S0", confidence=None, extract_status="ok")
        r.update(kw)
        if r["item_key"] is None:
            r["item_key"] = item_key_of(pk, seg_no, r["quote"])
        return r

    if resp is None:
        st["failed_schema"] += 1
        st["items"] += 1
        return [_row(nnn + 1, extract_status="failed_schema", summary=f"[L1 failed] {error or 'unknown error'}"[:500])], st
    items = resp.get("items") or []
    if not items:
        st["empty"] += 1
        note = _str_or_none(resp.get("segment_note"))
        if s3_hits([note], ctx["names"], ctx.get("first_names", ())):
            note = None
        return [_row(nnn, extract_status="empty", summary=(note or "")[:500] or None)], st

    seg_date = seg.get("segment_date")
    fallback_basis = "mail_date_fallback" if seg.get("date_inferred") else "segment_header"
    rows = []
    for it in items:
        nnn += 1
        st["items"] += 1
        if not isinstance(it, dict):
            st["failed_schema"] += 1
            rows.append(_row(nnn, extract_status="failed_schema", summary="[L1 failed] item is not an object"))
            continue
        summary, quote = _str_or_none(it.get("summary")), _str_or_none(it.get("quote"))
        stype = _str_or_none(it.get("statement_type"))
        stype = stype.lower() if stype else None
        # S3 先掃：命中就不讓內容落表
        if s3_hits([summary, quote], ctx["names"], ctx.get("first_names", ())):
            st["s3"] += 1
            st["failed_schema"] += 1
            rows.append(_row(nnn, extract_status="failed_schema", summary="[L1 failed] S3 hit in summary/quote",
                             statement_type=stype if stype in STATEMENT_TYPES else None))
            continue
        status = "ok"
        if stype not in STATEMENT_TYPES or not summary or not quote:
            status = "failed_schema"
        elif not quote_in_text(quote, seg.get("clean_text")):
            status = "failed_quote"
        # 受控詞表：不在表裡的 tag 移到 free_tags（§4.2 (4)）
        raw_tags = _str_list(it.get("topic_tags"), 20)
        tags = [t.lower() for t in raw_tags if t.lower() in TOPIC_TAGS][:MAX_TAGS]
        moved = [t for t in raw_tags if t.lower() not in TOPIC_TAGS]
        st["tags_moved"] += len(moved)
        free = _str_list(list(moved) + _str_list(it.get("free_tags"), MAX_FREE_TAGS), MAX_FREE_TAGS)
        if status == "ok" and not tags:
            st["no_topic"] += 1
        # 發言人：參與者表為準（§6.7）
        tok, side_tbl, reason = resolve_speaker(it.get("speaker_token"), ctx["pk_tokens"])
        if reason:
            st["speaker_dropped"][reason] = st["speaker_dropped"].get(reason, 0) + 1
        side_llm = _str_or_none(it.get("speaker_side"))
        side = side_tbl if tok else (side_llm.lower() if side_llm and side_llm.lower() in SIDES else "unknown")
        # 客戶：別名表正規化
        customer_raw = _str_or_none(it.get("customer_raw"))
        customer = normalize_customer(customer_raw, ctx["alias_exact"], ctx["alias_ordered"])
        if status == "ok" and customer is None:
            st["customer_null"] += 1
        # 型號 / app
        models = [m.upper() for m in _str_list(it.get("model_nos"), 20)]
        app = _str_or_none(it.get("app"))
        app = app.upper() if app else None
        basis = _str_or_none(it.get("app_basis"))
        if app not in APPS:
            app = _app_from_models(models)
            basis = "model_prefix" if app else "unknown"
        if basis not in APP_BASIS:
            basis = "unknown"
        # 日期（§6.5）；缺 → segment_date（§4.2 (8)）
        ev = _parse_date(it.get("event_date"))
        dbasis = _str_or_none(it.get("date_basis"))
        if ev is None:
            ev, dbasis = seg_date, fallback_basis
            st["date_fallback"] += 1
        elif dbasis not in DATE_BASIS:
            dbasis = "segment_header" if ev == seg_date else "inline_date"
        conf = it.get("confidence")
        conf = float(conf) if isinstance(conf, (int, float)) else None
        if conf is not None:
            conf = max(0.0, min(1.0, conf))
            if tok is None:
                conf = min(conf, SPEAKERLESS_CONF_CAP)
        st[status] += 1
        rows.append(_row(
            nnn, extract_status=status, statement_type=stype if stype in STATEMENT_TYPES else None,
            speaker_token=tok, speaker_side=side, customer_raw=customer_raw, customer=customer,
            app=app, app_basis=basis, model_nos=models, topic_tags=tags, topic_tag_primary=(tags[0] if tags else None),
            free_tags=free, summary=summary, quote=quote,
            event_date=ev, event_month=(ev.strftime("%Y-%m") if ev else None), date_basis=dbasis,
            time_refs=_json_list(it.get("time_refs")), numbers=_json_list(it.get("numbers")),
            terms_mentioned=_str_list(it.get("terms_mentioned"), 20),
            sensitivity=sensitivity_of(summary, quote), confidence=conf,
        ))
    return rows, st


def merge_stats(total: dict, part: dict) -> dict:
    for k, v in part.items():
        if isinstance(v, dict):
            total.setdefault(k, {})
            for kk, vv in v.items():
                total[k][kk] = total[k].get(kk, 0) + vv
        else:
            total[k] = total.get(k, 0) + v
    return total


In [ ]:
# [c06] llm_client
# LLM 接線：Azure Foundry 的 Anthropic Messages 端點，與 jobs/sales_report_mail/job_sales_brief_report [c04] 同一組
# secret / URL（規格 §5.0「沿 Sales Brief Job」）。用 requests（DBR 內建），不 %pip install。
# 參數：thinking 關閉、max_tokens 16000（§5.1 寫 4,000，2026-10-16 實測紀要段 JSON 會超過被截斷）、只要 JSON；system 開 prompt caching（§10）。
# 規格 §5.0 的 temperature 0 不送：claude-sonnet-5 回 400「temperature is deprecated for this model」（2026-10-16 實測），
# Claude 5 系列已移除取樣參數；輸出穩定度靠 prompt 與 item_key 比對（§9.2 重萃穩定率）。
# 重試：HTTP 429 / 5xx / 連線錯誤指數退避 3 次（§5.0）；4xx 其他直接回 error 不重試。JSON / schema 失敗的重試在 [c11]。
import requests

FOUNDRY_URL = "https://micenter.services.ai.azure.com/anthropic/v1/messages"
FOUNDRY_API_KEY = dbutils.secrets.get(scope="llm-api-keys", key="ms-foundry")
L1_MAX_TOKENS = 16000
HTTP_TIMEOUT_S = 180


def call_l1(messages: list[dict], system_prompt: str, model: str, cache: bool) -> dict:
    """messages = [{role, content}, …]（第一輪只有 user；重試時多帶上一輪 assistant ＋ 新 user）。
    回傳 {"text", "stop_reason", "usage": {input, output, cache_read, cache_write}, "latency_s", "error"}；
    error 非 None 代表端點層失敗（已重試），text 為 None。"""
    system = [{"type": "text", "text": system_prompt}]
    if cache:
        system[0]["cache_control"] = {"type": "ephemeral"}
    body = {"model": model, "max_tokens": L1_MAX_TOKENS, "system": system,
            "messages": messages, "thinking": {"type": "disabled"}}
    headers = {"x-api-key": FOUNDRY_API_KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"}
    t0 = time.time()
    err = None
    for attempt in range(3):
        try:
            resp = requests.post(FOUNDRY_URL, headers=headers, json=body, timeout=HTTP_TIMEOUT_S)
            if resp.status_code == 429 or resp.status_code >= 500:
                err = f"HTTP {resp.status_code}: {resp.text[:200]}"
            elif resp.status_code >= 400:
                return {"text": None, "stop_reason": None, "usage": {}, "latency_s": round(time.time() - t0, 2),
                        "error": f"HTTP {resp.status_code}: {resp.text[:300]}"}
            else:
                data = resp.json()
                text = "\n".join(b.get("text", "") for b in data.get("content", []) if b.get("type") == "text")
                u = data.get("usage") or {}
                usage = {"input": u.get("input_tokens", 0) or 0, "output": u.get("output_tokens", 0) or 0,
                         "cache_read": u.get("cache_read_input_tokens", 0) or 0,
                         "cache_write": u.get("cache_creation_input_tokens", 0) or 0}
                return {"text": text, "stop_reason": data.get("stop_reason"), "usage": usage,
                        "latency_s": round(time.time() - t0, 2), "error": None}
        except (requests.RequestException, ValueError) as e:
            err = f"{type(e).__name__}: {str(e)[:200]}"
        time.sleep(5 * (2 ** attempt))
    return {"text": None, "stop_reason": None, "usage": {}, "latency_s": round(time.time() - t0, 2), "error": err}


print("llm client ready:", FOUNDRY_URL.split("/anthropic")[0], "| model:", cfg["model"], "| cache:", cfg["prompt_cache"])


In [ ]:
# [c07] spark_transforms
# 純 DataFrame 函式：本批段落選取（已萃過的不重呼叫，§10 成本）、既有流水號、supersedes_item_id（§3.3）。不 cache（serverless）。
SEG_KEYS = ["pk_seq", "segment_no", "sub_no"]


def select_batch_segments(df_clean: DataFrame, df_items: DataFrame, prompt_version: str, retry_failed: bool) -> DataFrame:
    """可萃取段（content_type ∈ EXTRACTABLE_TYPES、dup_of IS NULL）扣掉「已萃過」的段。
    已萃過 = items 表有同 (pk, seg, sub) 且 prompt_version 相同、cleaner_version 與清洗表相同、superseded = false 的列；
    retry_failed = false 時 failed_* / empty 的列也算已萃過（不重打），true 時只有 ok / empty 算。"""
    done = df_items.where(~F.col("superseded") & (F.col("prompt_version") == F.lit(prompt_version)))
    if retry_failed:
        done = done.where(F.col("extract_status").isin("ok", "empty"))
    done = done.select(*SEG_KEYS, F.col("cleaner_version").alias("_cv")).distinct()
    return (
        df_clean.where(F.col("content_type").isin(*EXTRACTABLE_TYPES) & F.col("dup_of").isNull())
        .join(done, (df_clean.pk_seq == done.pk_seq) & (df_clean.segment_no == done.segment_no)
              & (df_clean.sub_no == done.sub_no) & (df_clean.cleaner_version == done._cv), "left_anti")
    )


def segment_offsets(df_items: DataFrame, df_batch: DataFrame) -> DataFrame:
    """本批每段既有條目的最大流水號 nnn（item_id 尾 3 碼）；新條目從 nnn+1 起編，item_id 才不會撞（「每次萃取新發」）。"""
    return (
        df_items.join(df_batch.select(*SEG_KEYS).distinct(), SEG_KEYS, "left_semi")
        .withColumn("_nnn", F.regexp_extract("item_id", r"-(\d{3})$", 1).cast("int"))
        .groupBy(*SEG_KEYS).agg(F.max("_nnn").alias("nnn_max"))
    )


def with_supersedes(df_new: DataFrame, df_items: DataFrame) -> DataFrame:
    """新條目對同段既有、未 superseded、status ok 的條目比 item_key；命中 → supersedes_item_id（取 item_id 最小者）。"""
    old = (
        df_items.where(~F.col("superseded") & (F.col("extract_status") == "ok"))
        .join(df_new.select(*SEG_KEYS).distinct(), SEG_KEYS, "left_semi")
        .groupBy(*SEG_KEYS, "item_key").agg(F.min("item_id").alias("_old_id"))
    )
    return (
        df_new.drop("supersedes_item_id")
        .join(old, SEG_KEYS + ["item_key"], "left")
        .withColumn("supersedes_item_id", F.when(F.col("extract_status") == "ok", F.col("_old_id")))
        .select(*df_new.columns)
    )


In [ ]:
# [c10] load_source
# 本批 = 清洗表可萃取段（inline_minutes / thread_reply、dup_of IS NULL）扣掉已萃過的（同 prompt_version ＋ 同 cleaner_version 且未
# superseded）。不另設水位；backfill 用 mail_date 窗。max_segments 是費用閘門，先切最舊的段（mail_date 順序），跑幾次就追上。
def fq(name: str) -> str:
    return f"{cfg['catalog']}.{cfg['schema']}.{name}"


fq_clean, fq_part = fq(f"s_meeting_record_clean_{cfg['unit']}"), fq(f"s_meeting_participants_{cfg['unit']}")
fq_items = fq(f"g_meeting_items_{cfg['unit']}")
fq_alias, fq_entity = fq("s_semantic_entity_alias"), fq("s_semantic_entity")
for t in (fq_clean, fq_part, fq_items, fq_alias, fq_entity):
    assert spark.catalog.tableExists(t), f"{t} 不存在：先跑 migration 001 / 002 與前兩支 w1_t1_mail、w1_t1_clean"

clean_tbl, items_tbl = spark.table(fq_clean), spark.table(fq_items)
if cfg["run_mode"] == "backfill":
    src = clean_tbl.where(F.col("mail_date").between(F.lit(cfg["date_from"]), F.lit(cfg["date_to"])))
    scope = f"backfill mail_date {cfg['date_from']} ~ {cfg['date_to']}"
else:
    src = clean_tbl
    scope = f"incremental：清洗表可萃取段中，items 表沒有 prompt_version {cfg['prompt_version']} 有效條目的段"
df_batch = select_batch_segments(src, items_tbl, cfg["prompt_version"], cfg["retry_failed"])
if cfg["max_segments"] > 0:
    df_batch = df_batch.orderBy("mail_date", *SEG_KEYS).limit(cfg["max_segments"])
    scope += f"；max_segments {cfg['max_segments']}"
df_batch = df_batch.select(*SEG_COLS)

# 別名（customer 正規化）：name / sb_brand、side = customer；entity_id 本身也算一個別名。小表，collect 到 driver。
ent_customer = spark.table(fq_entity).where(F.col("is_active") & (F.col("side") == "customer")).select("entity_id")
alias_rows = (
    spark.table(fq_alias).where(F.col("alias_type").isin("name", "sb_brand") & F.col("is_active"))
    .join(ent_customer, "entity_id").select("alias", "entity_id").distinct().collect()
)
aliases = [(r["alias"], r["entity_id"]) for r in alias_rows] + [(r["entity_id"], r["entity_id"]) for r in ent_customer.collect()]
alias_exact, alias_ordered = build_alias_index(aliases)
print(scope, "| customer aliases", len(alias_exact))


In [ ]:
# [c11] run_extract
# 段落 collect 到 driver 逐段呼叫 LLM（ThreadPool，concurrency 個併發）。這是刻意的例外：LLM 呼叫是 I/O 等待不是算力，
# 筆數受 max_segments 限制（全量 backfill 也只有千段級、每段 ≤ 12k 字），比 UDF 內打 HTTP 好控流量、好重試、好記 usage。
# 每段：user prompt → 呼叫 → 解析；JSON 失敗把上一輪輸出當 assistant turn、帶錯誤訊息再試 1 次（§5.0）；
# 輸出被 max_tokens 截斷則不回送截斷文字（白花 token），改成重新打一次並附 COMPACT_HINT。端點失敗的段不產生列（下次再撿）。
assert PROMPT_VERSION_IN_CELL == cfg["prompt_version"], (
    f"[c04] prompt 版本 {PROMPT_VERSION_IN_CELL} ≠ widget prompt_version {cfg['prompt_version']}（§2.4 第 4 點）")

segs = [r.asDict() for r in df_batch.collect()]
batch_pks = df_batch.select("pk_seq").distinct()
part_rows = spark.table(fq_part).select(*PART_COLS).join(batch_pks, "pk_seq", "left_semi").collect()
pk_tokens: dict[int, dict] = {}
pk_parts: dict[int, list] = {}
for r in part_rows:
    d = pk_tokens.setdefault(r["pk_seq"], {})
    e = d.setdefault(r["token"], {"side": None, "company": None, "kind": r["kind"], "is_forwarder": False})
    e["side"] = e["side"] or r["side"]
    e["company"] = e["company"] or r["company"]
    e["is_forwarder"] = e["is_forwarder"] or (r["role"] == "forwarder")
    pk_parts.setdefault(r["pk_seq"], []).append(r.asDict())
pk_names = {pk: name_variants_of(rows) for pk, rows in pk_parts.items()}  # 只在記憶體做 S3 掃描，不印
pk_first = {pk: first_name_res(rows) for pk, rows in pk_parts.items()}
offsets = {(r["pk_seq"], r["segment_no"], r["sub_no"]): r["nnn_max"]
           for r in segment_offsets(items_tbl, df_batch).collect()}
del part_rows, pk_parts
print(f"segments {len(segs)} | mails {len(pk_tokens)} | over_12k {sum(1 for s in segs if (s['clean_len'] or 0) > MAX_USER_CHARS)}")

usage_tot = {"input": 0, "output": 0, "cache_read": 0, "cache_write": 0, "calls": 0, "retries": 0}
latencies: list[float] = []


def _account(r: dict) -> None:
    usage_tot["calls"] += 1
    for k in ("input", "output", "cache_read", "cache_write"):
        usage_tot[k] += (r.get("usage") or {}).get(k, 0)
    latencies.append(r.get("latency_s") or 0.0)


def process_segment(seg: dict) -> dict:
    pk = seg["pk_seq"]
    ctx = {"pk_tokens": pk_tokens.get(pk, {}), "names": pk_names.get(pk, set()), "first_names": pk_first.get(pk, []),
           "alias_exact": alias_exact, "alias_ordered": alias_ordered,
           "prompt_version": cfg["prompt_version"], "model": cfg["model"],
           "nnn_offset": offsets.get((pk, seg["segment_no"], seg["sub_no"]), 0)}
    user_prompt = build_user_prompt(seg, ctx["pk_tokens"])
    messages = [{"role": "user", "content": user_prompt}]
    r1 = call_l1(messages, SYSTEM_PROMPT_L1, cfg["model"], cfg["prompt_cache"])
    _account(r1)
    if r1["error"]:
        return {"seg": seg, "rows": [], "stats": {}, "error": r1["error"], "truncated": False}
    truncated = r1["stop_reason"] == "max_tokens"
    obj, perr = (None, "output truncated at max_tokens") if truncated else parse_llm_json(r1["text"])
    if obj is None:
        usage_tot["retries"] += 1
        if truncated:
            messages = [{"role": "user", "content": user_prompt + COMPACT_HINT}]
        else:
            messages += [{"role": "assistant", "content": (r1["text"] or "").strip() or "(empty)"},
                         {"role": "user", "content": retry_user_prompt(perr)}]
        r2 = call_l1(messages, SYSTEM_PROMPT_L1, cfg["model"], cfg["prompt_cache"])
        _account(r2)
        if r2["error"]:
            return {"seg": seg, "rows": [], "stats": {}, "error": r2["error"], "truncated": truncated}
        if r2["stop_reason"] == "max_tokens":
            obj, perr, truncated = None, "output truncated at max_tokens (retry)", True
        else:
            obj, perr = parse_llm_json(r2["text"])
    rows, st = build_item_rows(obj, perr, seg, ctx)
    return {"seg": seg, "rows": rows, "stats": st, "error": None, "truncated": truncated}


results: list[dict] = []
if segs:
    with ThreadPoolExecutor(max_workers=cfg["concurrency"]) as ex:
        results = list(ex.map(process_segment, segs))
all_rows = [row for r in results for row in r["rows"]]
errors = [(r["seg"]["pk_seq"], r["seg"]["segment_no"], r["error"]) for r in results if r["error"]]
n_new_rows = len(all_rows)

items_schema = spark.table(fq_items).schema
df_new = spark.createDataFrame([tuple(row.get(f.name) for f in items_schema.fields) for row in all_rows], items_schema)
df_new = (
    with_supersedes(df_new, items_tbl)
    .withColumn("extracted_at", F.current_timestamp())
    .select(*items_schema.fieldNames())
)
df_new.createOrReplaceTempView("_w1_items_batch")
print(f"rows {n_new_rows} | calls {usage_tot['calls']} | retries {usage_tot['retries']} | endpoint errors {len(errors)}")


In [ ]:
# [c12] gates
# 關卡（§4.2 / §10 告警）：端點全掛 → 不寫半批；S3：LLM 輸出命中的列內容已在 [c05] 清空（寫進表的 S3 = 0 由程式保證，[c30] 再查），
# 命中率 > 5% 才停、否則 WARN（D25）；schema 失敗 > 10%、quote 失敗 > 15% → 停（§10 task fail 門檻；
# > 5% / > 10% 先 WARN，§4.2）。只印 log：customer NULL 比例（> 20% 代表別名表要補）、發言人被置 NULL 的原因、tag 移到 free_tags、
# 日期補值、超長段、token 用量與費用估算（§10.1 要求 Phase 1 實測）。
def pct(n, d) -> float | None:
    return round(n / d, 3) if d else None


tot: dict = {}
for r in results:
    merge_stats(tot, r["stats"])
n_seg, n_err = len(segs), len(errors)
n_items = tot.get("items", 0)
schema_fail, quote_fail = pct(tot.get("failed_schema", 0), n_items), pct(tot.get("failed_quote", 0), n_items)
cost = (usage_tot["input"] * PRICE_PER_MTOK["input"] + usage_tot["output"] * PRICE_PER_MTOK["output"]
        + usage_tot["cache_read"] * PRICE_PER_MTOK["cache_read"] + usage_tot["cache_write"] * PRICE_PER_MTOK["cache_write"]) / 1e6
log = {
    "scope": scope, "prompt_version": cfg["prompt_version"], "model": cfg["model"],
    "segments": n_seg, "segments_endpoint_error": n_err,
    "segments_over_12k": sum(1 for s in segs if (s["clean_len"] or 0) > MAX_USER_CHARS),
    "segments_empty": tot.get("empty", 0), "segments_truncated": sum(1 for r in results if r.get("truncated")),
    "items": n_items, "ok": tot.get("ok", 0), "failed_schema": tot.get("failed_schema", 0),
    "failed_quote": tot.get("failed_quote", 0), "s3_hits": tot.get("s3", 0),
    "schema_fail_rate": schema_fail, "quote_fail_rate": quote_fail,
    "items_per_segment": pct(tot.get("ok", 0), n_seg - n_err),
    "customer_null_rate": pct(tot.get("customer_null", 0), tot.get("ok", 0)),
    "speaker_dropped": tot.get("speaker_dropped", {}), "tags_moved_to_free": tot.get("tags_moved", 0),
    "items_without_topic": tot.get("no_topic", 0), "date_fallback": tot.get("date_fallback", 0),
    "llm": {**usage_tot, "latency_p50_s": (sorted(latencies)[len(latencies) // 2] if latencies else None),
            "est_cost_usd": round(cost, 3)},
}
print(json.dumps(log, ensure_ascii=False, default=str, indent=1))
if errors:
    print("端點失敗的段（最多 10）：", errors[:10])
if n_seg == 0:
    print("本批 0 段：沒有新的可萃取段，後面 cell 不寫表。")
else:
    assert n_err < n_seg, "LLM 端點整體不可用：全部呼叫失敗，不寫半批（§4.2）"
    assert n_err / n_seg <= 0.5, f"端點失敗 {n_err}/{n_seg} 超過一半：先查端點再跑（失敗的段下次會再撿）"
if tot.get("s3", 0):
    print(f"WARN S3：LLM 輸出 {tot['s3']} 條含 email / 參與者姓名，已清空內容標 failed_schema；多的話回頭看 [c04] 人名規則")
    assert tot["s3"] / max(n_items, 1) <= 0.05, "S3 命中率 > 5%：prompt 的人名規則沒生效，先修再跑"
if n_items:
    if schema_fail > 0.05:
        print(f"WARN schema 失敗率 {schema_fail} > 5%（§4.2 關卡）")
    if quote_fail > 0.10:
        print(f"WARN quote 失敗率 {quote_fail} > 10%（§4.2 關卡）")
    if (log["customer_null_rate"] or 0) > 0.20:
        print(f"WARN customer NULL 比例 {log['customer_null_rate']} > 20%：別名表要補（看 dry_run 樣本的 customer_raw）")
    assert schema_fail <= 0.10, f"schema 失敗率 {schema_fail} > 10%（§10 告警門檻）：調 [c04] prompt"
    assert quote_fail <= 0.15, f"quote 失敗率 {quote_fail} > 15%（§10 告警門檻）：調 [c04] quote 規則"
if cfg["dry_run"] and n_new_rows:
    print("\ndry_run = true：不寫表。樣本（已 token 化，無人名）：")
    df_new.select("item_id", "extract_status", "statement_type", "speaker_token", "speaker_side", "customer_raw", "customer",
                  "app", "app_basis", "topic_tags", "model_nos", "event_date", "date_basis", "sensitivity", "confidence",
                  F.substring("summary", 1, 160).alias("summary_head"), F.substring("quote", 1, 120).alias("quote_head"),
                  ).orderBy("item_id").limit(15).show(truncate=100, vertical=True)
    print("customer 對不上的原名（最多 20）：",
          sorted({r["customer_raw"] for r in all_rows if r["extract_status"] == "ok" and r["customer"] is None and r["customer_raw"]})[:20])


In [ ]:
# [c20] write_items
# 兩步：(1) 本批有產出的段，既有未 superseded 的條目全標 superseded = true（重萃不刪，§3.3 / §10 冪等）；
# (2) 新條目 append（item_id 以既有流水號接著編，不會撞）。端點失敗的段沒有新列，舊條目不動，下次 incremental 再撿。
# 不用 MERGE 做 append：item_id 每次新發，沒有「同鍵更新」的情境。
if not cfg["dry_run"] and n_new_rows > 0:
    spark.sql(f"""
        MERGE INTO {fq_items} t
        USING (SELECT DISTINCT pk_seq, segment_no, sub_no FROM _w1_items_batch) s
        ON t.pk_seq = s.pk_seq AND t.segment_no = s.segment_no AND t.sub_no = s.sub_no AND t.superseded = false
        WHEN MATCHED THEN UPDATE SET t.superseded = true
    """)
    spark.table("_w1_items_batch").write.mode("append").saveAsTable(fq_items)
    print(f"superseded old items for batch segments, appended {n_new_rows} rows →", fq_items)
else:
    print("skip write（dry_run 或本批 0 列）")


In [ ]:
# [c30] check_target
# 寫入後驗證：本批段的有效（未 superseded）條目 = 本批列數、只有一個 prompt_version；item_id 無重複；
# 有效條目 summary / quote 沒有 email 樣式（S3，SQL 再查一次）；supersedes_item_id 指向的條目存在。
if not cfg["dry_run"] and n_new_rows > 0:
    all_items = spark.table(fq_items)
    keys = spark.table("_w1_items_batch").select(*SEG_KEYS).distinct()
    active = all_items.join(keys, SEG_KEYS, "left_semi").where(~F.col("superseded"))
    email_rx = r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+\.[A-Za-z]{2,}"
    chk = {
        "batch_active_rows": active.count(),
        "expected_rows": n_new_rows,
        "dup_item_id": all_items.groupBy("item_id").count().where("count > 1").count(),
        "mixed_prompt_version_segments": active.groupBy(*SEG_KEYS).agg(F.countDistinct("prompt_version").alias("n")).where("n > 1").count(),
        "email_in_active": active.where(F.col("summary").rlike(email_rx) | F.col("quote").rlike(email_rx)).count(),
        "dangling_supersedes": (
            active.where(F.col("supersedes_item_id").isNotNull()).select(F.col("supersedes_item_id").alias("item_id"))
            .join(all_items.select("item_id"), "item_id", "left_anti").count()
        ),
        "table_total": all_items.count(),
        "table_active_ok": all_items.where(~F.col("superseded") & (F.col("extract_status") == "ok")).count(),
    }
    print(chk)
    assert chk["batch_active_rows"] == chk["expected_rows"], "本批段的有效條目數 ≠ 本批列數（MERGE 或 append 沒做完）"
    assert chk["dup_item_id"] == 0, "item_id 重複"
    assert chk["mixed_prompt_version_segments"] == 0, "同一段同時有兩個 prompt_version 的有效條目"
    assert chk["email_in_active"] == 0, "有效條目殘留 email（S3 硬規則）"
    assert chk["dangling_supersedes"] == 0, "supersedes_item_id 指向不存在的條目"
